## Data Cleaning and Preparation
What does this dataset do 
- this notebook cleans the data and creates a csv file that will be ready for: 

a. NASA-TLX data analysis 
- This will be the document analyzing the perceived workload of tasks done. 
    
b. Simulation data analysis
- This will be the document analyzing the statistics and the actions done by the participants during the simulation. 

Important note: 
csv files for nasa-tlx should be in 1 folder and csv files for simulation should be in a separate folder

#### Data Description


### Simulation 
| Column Name        | Description                                                  |
|--------------------|--------------------------------------------------------------|
| **timestamp** | recorded time when an occurence occured |
| **phase** | classification of drive |
| **scenario** | Type of occurence that happened|
| **speed_kmh** | recorded speed the participant was going |
| **event** | the violation or occurence type |
| **details** | Split into Violation and Location |
| **Violation** | Specific Violation acquired (?) | 
| **Location** | Exact coordinates the violation occured. This differs every time. |

### NASA TLX 
| Column Name        | Description                                                  |
|--------------------|--------------------------------------------------------------|
| **Participant**   | Participant number for tracking, acts as a unique identifier of a participant |
| **Task Name** | Type of alerts that the participant took (Baseline, AO - Audio Only, HO - Haptic Only, AH - Audio and Haptic) |
| **Mental Demand**| Mental and perceptual activity |
| **Physical Demand**| Physical activity required | 
| **Temporal Demand** | Time pressure according to the pace |
| **Performance** | Successfulness in completing the task |
| **Effort** | 
| **Frustration**| |


#### Data Collection & Methodology 
**NASATLX**
This dataset came from the NASATLX site. The participants were asked to answer the same exact questions. This was done every after drive to gauge their experience immediately after the drive. Upon completing the whole 4 drives, the csv file was generated. 

**Simulation**
This dataset came from the log tracker of the CARLA application, which records the corresponding type of drive and the violations committed. When a violation was done, the coordinates, 


#### Biases and Limitations 

#### Sources and Credits
NASA-TLX site: https://nasa-tlx-calculator.vercel.app/

**Reminders**
- make sure there are 2 separate folders > 1 each for the respective data (1 csv folder for Sim, 1 csv folder for NASATLX)

### Preprocessing

In [96]:
import pandas as pd
import numpy as np
import glob
import re
import json
 

In [97]:
# load all csv files 
# change inside "" to the folder path that contains all the simulation drives csv files. 
files = glob.glob("/Users/karencafe/Desktop/ThesisTest/SIMCSV/*.csv")

df_simulation = pd.concat(
    (pd.read_csv(f) for f in files),
    ignore_index=True
)

display(df_simulation)


,timestamp,phase,scenario,speed_kmh,event,details
0,1.771922e+09,4,START,0.00,PHASE_START,Assisted-AH
1,1.771923e+09,4,START,0.00,PHASE_START,Assisted-AH
2,1.771923e+09,4,WEBSOCKET,0.00,WS_SEND,"{""phase"": 4, ""speed"": 0.0, ""speed_limit"": 30, ..."
3,1.771923e+09,4,WEBSOCKET,0.00,WS_SEND,"{""phase"": 4, ""speed"": 0.0, ""speed_limit"": 30, ..."
4,1.771923e+09,4,WEBSOCKET,0.00,WS_SEND,"{""phase"": 4, ""speed"": 0.0, ""speed_limit"": 30, ..."
...,...,...,...,...,...,...
748694,1.771682e+09,3,WEBSOCKET,0.00,WS_SEND,"{""phase"": 3, ""speed"": 0.0, ""speed_limit"": 30, ..."
748695,1.771682e+09,3,WEBSOCKET,0.00,WS_SEND,"{""phase"": 3, ""speed"": 0.0, ""speed_limit"": 30, ..."
748696,1.771682e+09,3,WEBSOCKET,0.00,WS_SEND,"{""phase"": 3, ""speed"": 0.0, ""speed_limit"": 30, ..."
748697,1.771682e+09,3,STOP,674.87,PHASE_STOP,Assisted-H


In [98]:
df_simulation["timestamp"] = pd.to_numeric(df_simulation["timestamp"], errors="coerce")
df_simulation["speed_kmh"] = pd.to_numeric(df_simulation["speed_kmh"], errors="coerce")

In [99]:
# removing empty HUD alerts
df_simulation = df_simulation[
    ~(
        (df_simulation["event"] == "HUD_CUE") &
        (df_simulation["details"].str.strip() == "|")
    )
]

In [100]:
# removing speed with 0-0.1 -> too many and insignificant 
df_simulation = df_simulation[df_simulation["speed_kmh"] > 0.1]

In [101]:
df_simulation

,timestamp,phase,scenario,speed_kmh,event,details
1300,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,..."
1301,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,..."
1302,1.771924e+09,4,WEBSOCKET,0.71,WS_SEND,"{""phase"": 4, ""speed"": 0.71, ""speed_limit"": 30,..."
1303,1.771924e+09,4,WEBSOCKET,1.17,WS_SEND,"{""phase"": 4, ""speed"": 1.17, ""speed_limit"": 30,..."
1304,1.771924e+09,4,WEBSOCKET,1.19,WS_SEND,"{""phase"": 4, ""speed"": 1.19, ""speed_limit"": 30,..."
...,...,...,...,...,...,...
748642,1.771682e+09,3,WEBSOCKET,3.53,WS_SEND,"{""phase"": 3, ""speed"": 3.53, ""speed_limit"": 30,..."
748643,1.771682e+09,3,WEBSOCKET,3.53,WS_SEND,"{""phase"": 3, ""speed"": 3.53, ""speed_limit"": 30,..."
748644,1.771682e+09,3,WEBSOCKET,3.41,WS_SEND,"{""phase"": 3, ""speed"": 3.41, ""speed_limit"": 30,..."
748697,1.771682e+09,3,STOP,674.87,PHASE_STOP,Assisted-H


In [102]:
## SET SPEED_LIMIT column
def extract_speed_limit(text):
    if pd.isna(text):
        return None
    
    text = str(text)
    
    # Case 1: JSON-like
    if text.strip().startswith("{"):
        try:
            data = json.loads(text)
            return float(data.get("speed_limit"))
        except:
            return None
    
    # Case 2: Plain text
    match = re.search(r"limit\s([\d.]+)", text)
    if match:
        return float(match.group(1))
    
    return None

df_simulation["speed_limit"] = df_simulation["details"].apply(extract_speed_limit)

In [103]:
display(df_simulation[df_simulation["phase"] == 2].tail(20))

,timestamp,phase,scenario,speed_kmh,event,details,speed_limit
713889,1.771679e+09,2,SPEED_LIMIT,61.60,REACTION,action=REDUCE_THROTTLE|time=1.74s|control={'th...,NaN
713890,1.771679e+09,2,SPEED_LIMIT,62.28,SPEED_VIOLATION,Speed 62.28 km/h > limit 30.00 km/h at Locatio...,30.0
713891,1.771679e+09,2,SPEED_LIMIT,37.00,REACTION,action=VIOLATION_RESOLVED|time=3.27s|reaction_...,NaN
713892,1.771679e+09,2,SPEED_LIMIT,38.69,REACTION,action=REDUCE_THROTTLE|time=0.10s|control={'th...,NaN
713893,1.771679e+09,2,SPEED_LIMIT,38.86,SPEED_VIOLATION,Speed 38.86 km/h > limit 30.00 km/h at Locatio...,30.0
713894,1.771679e+09,2,SPEED_LIMIT,36.94,REACTION,action=VIOLATION_RESOLVED|time=0.42s|reaction_...,NaN
713895,1.771679e+09,2,SPEED_LIMIT,52.57,REACTION,action=REDUCE_THROTTLE|time=2.49s|control={'th...,NaN
713896,1.771679e+09,2,SPEED_LIMIT,52.57,SPEED_VIOLATION,Speed 52.57 km/h > limit 30.00 km/h at Locatio...,30.0
713897,1.771679e+09,2,SPEED_LIMIT,35.48,REACTION,action=VIOLATION_RESOLVED|time=3.63s|reaction_...,NaN
713898,1.771679e+09,2,TRAFFIC_LIGHT,31.68,RED_LIGHT_VIOLATION,TrafficLight Violation at Location(x=68.271347...,NaN


In [104]:
display(df_simulation['speed_limit'].unique())

array([30., nan, 90., 60.])

In [105]:
# GET LOCATION
def extract_location(text):
    if pd.isna(text):
        return pd.Series([None, None])
    
    text = str(text)

    # Case 1: JSON-like
    if text.strip().startswith("{"):
        try:
            data = json.loads(text)
            loc = data.get("location") or data.get("Location")
            if loc:
                return pd.Series([loc.get("x"), loc.get("y")])
        except:
            pass

    # Case 2: Plain text (CARLA format)
    match = re.search(r"Location.*?x[:=]\s*([\-\d.]+).*?y[:=]\s*([\-\d.]+)", text)
    if match:
        return pd.Series([float(match.group(1)), float(match.group(2))])

    return pd.Series([None, None])

df_simulation[["Location_X", "Location_Y"]] = df_simulation["details"].apply(extract_location)

In [106]:
# Mark all = 0; start violation = 1; end violation = -1
#df_simulation["violation_flag"] = 0

# Start violation when speed exceeds limit + 7
#df_simulation.loc[
#    df_simulation["speed_kmh"] > df_simulation["speed_limit"] + 7,
#    "violation_flag"
#] = 1

# Keep this
#df_simulation.loc[
#    df_simulation["details"].str.contains("VIOLATION_RESOLVED", na=False),
#    "violation_flag"
#] = -1

In [107]:
# marking the start and marking resolved. 
df_simulation["violation_start"] = df_simulation["details"].str.contains('"overspeed": true', na=False)
df_simulation["violation_end"] = df_simulation["details"].str.contains("VIOLATION_RESOLVED", na=False)

In [124]:
df_simulation["violation_state"] = 0

# checks if violation on going 
active = False
states = []

for start, end in zip(df_simulation["violation_start"], df_simulation["violation_end"]):

    if start and not active:
        active = True # marks the start of violation 

    if active:
        states.append(1)
    else:
        states.append(0) 

    if end:
        active = False # violation end 

df_simulation["violation_state"] = states

# start = rows from violation_start = true
# end = until violation_end = true
# if start is true and active is false => make active true (start counting the rows as 1)
# while active is true => append 1 to the array
# if violation_end becomes true => active becomes false AFTER this row, so the loop will append 0 starting from the next row

In [110]:
df_simulation.columns

Index(['timestamp', 'phase', 'scenario', 'speed_kmh', 'event', 'details',
       'speed_limit', 'Location_X', 'Location_Y', 'violation_start',
       'violation_end', 'violation_state'],
      dtype='object')

In [115]:
# checking lang 
display(df_simulation[df_simulation["event"].isin(["SPEED_VIOLATION","REACTION"])].head(5))

,timestamp,phase,scenario,speed_kmh,event,details,speed_limit,Location_X,Location_Y,violation_start,violation_end,violation_state,event_start,event_id
2591,1.771924e+09,4,SPEED_LIMIT,41.58,REACTION,action=REDUCE_THROTTLE|time=0.76s|control={'th...,NaN,NaN,NaN,False,False,1,0,1
2603,1.771924e+09,4,SPEED_LIMIT,41.77,SPEED_VIOLATION,Speed 41.77 km/h > limit 30.00 km/h at Locatio...,30.0,-97.75,12.717696,False,False,1,0,1
2604,1.771924e+09,4,SPEED_LIMIT,36.30,REACTION,action=VIOLATION_RESOLVED|time=1.33s|reaction_...,NaN,NaN,NaN,False,True,1,0,1
4516,1.771924e+09,4,TRAFFIC_LIGHT,2.22,REACTION,action=NO_REACTION|time=3.01s|control={'thrott...,NaN,NaN,NaN,False,False,0,0,0
7549,1.771924e+09,4,TRAFFIC_LIGHT,36.42,REACTION,action=NO_REACTION|time=3.04s|control={'thrott...,NaN,NaN,NaN,False,False,0,0,0


In [112]:
# Checks when a speeding event starts 
df_simulation["event_start"] = (
    (df_simulation["violation_state"] == 1) &
    (df_simulation["violation_state"].shift(1) != 1)
).astype(int)

In [113]:
# sets an identifier for every start of an event, then increments 
df_simulation["event_id"] = df_simulation["event_start"].cumsum()
df_simulation.loc[df_simulation["violation_state"] == 0, "event_id"] = 0

In [116]:
# drops irrelevant columns (can be kept if wanted- does not harm data, but cleaner view)
df_simulation.drop("event_start", axis=1, inplace=True)
df_simulation.drop("scenario", axis=1, inplace=True)

In [117]:
df_simulation.columns

Index(['timestamp', 'phase', 'speed_kmh', 'event', 'details', 'speed_limit',
       'Location_X', 'Location_Y', 'violation_start', 'violation_end',
       'violation_state', 'event_id'],
      dtype='object')

In [123]:
#df_simulation['event_id'].unique()

In [120]:
# checking lang if action = slow down and action = resolved has same event_id
df_simulation.loc[df_simulation["event_id"] == 294, ["violation_state", "details", "event", "event_id"]]

,violation_state,details,event,event_id
424433,1,"{""phase"": 2, ""speed"": 37.06, ""speed_limit"": 30...",WS_SEND,294
424434,1,"{""phase"": 2, ""speed"": 37.13, ""speed_limit"": 30...",WS_SEND,294
424435,1,"{""phase"": 2, ""speed"": 37.13, ""speed_limit"": 30...",WS_SEND,294
424436,1,"{""phase"": 2, ""speed"": 37.13, ""speed_limit"": 30...",WS_SEND,294
424437,1,"{""phase"": 2, ""speed"": 37.26, ""speed_limit"": 30...",WS_SEND,294
424438,1,"{""phase"": 2, ""speed"": 37.32, ""speed_limit"": 30...",WS_SEND,294
424439,1,"{""phase"": 2, ""speed"": 37.41, ""speed_limit"": 30...",WS_SEND,294
424440,1,"{""phase"": 2, ""speed"": 37.4, ""speed_limit"": 30,...",WS_SEND,294
424441,1,"{""phase"": 2, ""speed"": 37.4, ""speed_limit"": 30,...",WS_SEND,294
424442,1,"{""phase"": 2, ""speed"": 37.46, ""speed_limit"": 30...",WS_SEND,294


In [121]:
df_simulation['event'].unique()

array(['WS_SEND', 'YELLOW_LIGHT_PASS', 'REACTION', 'SPEED_VIOLATION',
       'RED_LIGHT_VIOLATION', 'PHASE_STOP', 'HUD_CUE'], dtype=object)

In [122]:
df_simulation.to_csv("cleaned_Sim.csv", index=False)

# NASA TLX CLEANING

In [ ]:
files = glob.glob("/Users/karencafe/Desktop/ThesisTest/NASACSV/*.csv")
# change to the path of the NASATLX folder containing all the csv files

df2 = pd.concat(
    (pd.read_csv(f) for f in files),
    ignore_index=True
)

display(df2)


,Participant,Task Id,Task Name,Mental Demand,Physical Demand,Temporal Demand,Performance,Effort,Frustration,Raw Score,MD Weight,PD Weight,TD Weight,PF Weight,EF Weight,FR Weight,Weighted Score
0,8,1,AH,70,60,40,70,60,80,63.333333,-1,-1,-1,-1,-1,-1,-1
1,8,2,NC,80,70,70,70,70,70,71.666667,-1,-1,-1,-1,-1,-1,-1
2,8,3,AO,80,70,70,70,70,70,71.666667,-1,-1,-1,-1,-1,-1,-1
3,8,4,HO,80,70,70,70,70,70,71.666667,-1,-1,-1,-1,-1,-1,-1
4,9,1,NC,20,40,40,60,60,40,43.333333,-1,-1,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,11,5,NC,5,15,5,65,50,0,23.333333,-1,-1,-1,-1,-1,-1,-1
68,5,1,AH,80,80,60,20,20,20,46.666667,-1,-1,-1,-1,-1,-1,-1
69,5,2,NC,80,80,60,10,10,10,41.666667,-1,-1,-1,-1,-1,-1,-1
70,5,3,HO,90,90,70,20,20,20,51.666667,-1,-1,-1,-1,-1,-1,-1


In [ ]:
df2 = df2.drop(columns=["Task Id", "MD Weight", "PD Weight", "TD Weight", "PF Weight", "EF Weight", "FR Weight", "Weighted Score"])

display(df2)

,Participant,Task Name,Mental Demand,Physical Demand,Temporal Demand,Performance,Effort,Frustration,Raw Score
0,8,AH,70,60,40,70,60,80,63.333333
1,8,NC,80,70,70,70,70,70,71.666667
2,8,AO,80,70,70,70,70,70,71.666667
3,8,HO,80,70,70,70,70,70,71.666667
4,9,NC,20,40,40,60,60,40,43.333333
...,...,...,...,...,...,...,...,...,...
67,11,NC,5,15,5,65,50,0,23.333333
68,5,AH,80,80,60,20,20,20,46.666667
69,5,NC,80,80,60,10,10,10,41.666667
70,5,HO,90,90,70,20,20,20,51.666667


In [ ]:
df2["Raw Score"] = df2["Raw Score"].round(2)
display(df2.head(3))

,Participant,Task Name,Mental Demand,Physical Demand,Temporal Demand,Performance,Effort,Frustration,Raw Score
0,8,AH,70,60,40,70,60,80,63.33
1,8,NC,80,70,70,70,70,70,71.67
2,8,AO,80,70,70,70,70,70,71.67


In [ ]:
# Makes sure no mislabeled Task Name
print(df2['Task Name'].unique())

['AH' 'NC' 'AO' 'HO']


Clean task name 
- NC = no cue / baseline 
- AO = Audio Only 
- HO = Haptic Only 
- AH = Audio and Haptic / both Combined 

In [ ]:
df2.columns = df2.columns.str.strip()          # removes hidden spaces
df2.columns = df2.columns.str.replace(" ", "_")

In [ ]:
df2.to_csv("cleaned_NASATLX.csv", index=False)